# Figure 2: Finite-sample calculation of K_c

This notebook computes the finite-sample version of Eq. (28) in the
manuscript. It computes the empirical tangent-projection factor and
the empirical intrinsic-drift resolvent average, and checks whether
their product converges to the corresponding integral value.

The simulation functions use the ambient dimension \(n\) for
\(S^{n-1}\subset\mathbb{R}^n\). The manuscript uses
\(D=\dim M=n-1\), so the saved data and plotted labels report the
manifold dimension.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.lines import Line2D

# Set the experiment root for runs started from the notebook directory or the
# experiment directory.
HERE = Path.cwd().resolve()
EXPERIMENT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
SRC_DIR = EXPERIMENT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import DATA_DIR, FIGURE_DIR, ensure_directories
from plotting import set_paper_style, save_figure
from manifold_sync import *

ensure_directories()
set_paper_style()


def manuscript_palette(labels):
    # Return the blue-green manuscript palette used for dimension-like groups.
    labels = list(labels)
    colors = sns.color_palette("viridis", n_colors=len(labels))
    return dict(zip(labels, colors))

In [ ]:
# Ambient dimensions n=2,4,6,8,10 correspond to manifold
# dimensions D=1,3,5,7,9 in the manuscript notation.
AMBIENT_DIMENSIONS = [2, 4, 6, 8, 10]
DELTAS = np.linspace(0.04, 1.20, 40)
N_VALUES = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
SEEDS = range(30)
EPSILON_FACTOR = 1.0e-2

planned_rows = (
    len(AMBIENT_DIMENSIONS)
    * len(DELTAS)
    * len(N_VALUES)
    * len(SEEDS)
)
print(f"Planned finite-sample threshold evaluations: {planned_rows:,}")

In [ ]:
def stratified_lorentzian_sample(n, delta, rng):
    # Return a randomized stratified sample from a Lorentzian density.
    q = (np.arange(n) + rng.random()) / n
    return delta * np.tan(np.pi * (q - 0.5))


records = []
for ambient_dimension in AMBIENT_DIMENSIONS:
    manifold_dimension = ambient_dimension - 1
    kappa_theory = kappa_sphere(ambient_dimension)
    for n in N_VALUES:
        for seed in SEEDS:
            rng = np.random.default_rng(
                10_000 * ambient_dimension + 100 * int(np.log2(n)) + seed
            )
            points = random_sphere_points(rng, n, ambient_dimension)
            kappa_sample = np.mean(1.0 - points[:, 0] ** 2)
            for delta in DELTAS:
                omega = stratified_lorentzian_sample(n, float(delta), rng)
                epsilon = EPSILON_FACTOR * float(delta)
                resolvent_sample = np.mean(epsilon / (epsilon**2 + omega**2))
                susceptibility_sample = kappa_sample * resolvent_sample
                Kc_sample = 1.0 / susceptibility_sample
                Kc_epsilon_theory = 1.0 / spectral_susceptibility_hopf(
                    D=ambient_dimension,
                    delta=float(delta),
                    distribution="lorentzian",
                    epsilon_factor=EPSILON_FACTOR,
                )
                Kc_neutral_theory = hopf_threshold_theory(
                    ambient_dimension,
                    float(delta),
                )
                records.append(
                    {
                        "ambient_dimension": ambient_dimension,
                        "manifold_dimension": manifold_dimension,
                        "Manifold dimension": f"D = {manifold_dimension}",
                        "N": n,
                        "seed": seed,
                        "delta": float(delta),
                        "epsilon_factor": EPSILON_FACTOR,
                        "kappa_sample": kappa_sample,
                        "kappa_theory": kappa_theory,
                        "kappa_ratio": kappa_sample / kappa_theory,
                        "resolvent_sample": resolvent_sample,
                        "susceptibility_sample": susceptibility_sample,
                        "Kc_sample": Kc_sample,
                        "Kc_epsilon_theory": Kc_epsilon_theory,
                        "Kc_neutral_theory": Kc_neutral_theory,
                        "threshold_ratio": Kc_sample / Kc_epsilon_theory,
                        "neutral_ratio": Kc_sample / Kc_neutral_theory,
                    }
                )

finite_sample = pd.DataFrame.from_records(records)
finite_sample["N_inv_sqrt"] = 1.0 / np.sqrt(finite_sample["N"])
finite_sample["abs_relative_error"] = np.abs(finite_sample["threshold_ratio"] - 1.0)

geometry_sample = finite_sample[
    [
        "ambient_dimension",
        "manifold_dimension",
        "Manifold dimension",
        "N",
        "seed",
        "kappa_sample",
        "kappa_theory",
        "kappa_ratio",
        "N_inv_sqrt",
    ]
].drop_duplicates()

finite_sample.to_csv(DATA_DIR / "fig02_finite_sample_threshold.csv", index=False)
geometry_sample.to_csv(DATA_DIR / "fig02_finite_sample_geometry.csv", index=False)

finite_sample.head()

In [ ]:
TITLE_SIZE = 9.5
LABEL_SIZE = 9.5
TICK_SIZE = 8.5
LEGEND_SIZE = 7.5
LEGEND_TITLE_SIZE = 8.5
PANEL_LABEL_SIZE = 12

fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.25))

legend_style = dict(
    frameon=True,
    fontsize=LEGEND_SIZE,
    title_fontsize=LEGEND_TITLE_SIZE,
    borderpad=0.35,
    labelspacing=0.25,
    handlelength=1.4,
    handletextpad=0.45,
)
dimension_order = sorted(finite_sample["manifold_dimension"].unique())
dimension_labels = [f"D = {D}" for D in dimension_order]
dimension_palette = manuscript_palette(dimension_labels)

sns.lineplot(
    data=geometry_sample,
    x="N_inv_sqrt",
    y="kappa_ratio",
    hue="Manifold dimension",
    hue_order=dimension_labels,
    palette=dimension_palette,
    marker="o",
    markersize=4,
    lw=1.25,
    ax=axes[0],
    errorbar=("ci", 95),
)
axes[0].axhline(1.0, color="black", lw=0.8)
axes[0].set_xlim(left=0.0)
axes[0].set_title(r"Empirical $\kappa_N$")
axes[0].set_xlabel(r"$N^{-1/2}$")
axes[0].set_ylabel(r"$\kappa_N/\kappa(S^D)$")
axes[0].legend(title="Manifold dimension", loc="best", ncol=1, **legend_style)

largest_n = max(N_VALUES)
threshold_large = finite_sample[finite_sample["N"] == largest_n].copy()
sns.lineplot(
    data=threshold_large,
    x="delta",
    y="Kc_sample",
    hue="Manifold dimension",
    hue_order=dimension_labels,
    palette=dimension_palette,
    marker="o",
    markersize=3,
    lw=1.25,
    errorbar=("ci", 95),
    ax=axes[1],
)

for manifold_dimension in dimension_order:
    ambient_dimension = manifold_dimension + 1
    dimension_label = f"D = {manifold_dimension}"
    axes[1].plot(
        DELTAS,
        [
            1.0
            / spectral_susceptibility_hopf(
                D=ambient_dimension,
                delta=float(delta),
                distribution="lorentzian",
                epsilon_factor=EPSILON_FACTOR,
            )
            for delta in DELTAS
        ],
        color=dimension_palette[dimension_label],
        lw=0.9,
        ls="--",
        alpha=0.85,
    )
axes[1].set_title(r"$K_{c,N}^{(\epsilon)}$ versus $\Delta$")
axes[1].set_xlabel(r"Lorentzian width parameter $\Delta$")
axes[1].set_ylabel(r"$K_{c,N}^{(\epsilon)}$")
axes[1].legend(title="Manifold dimension", loc="upper left", ncol=1, **legend_style)

sns.lineplot(
    data=finite_sample,
    x="N",
    y="threshold_ratio",
    hue="Manifold dimension",
    hue_order=dimension_labels,
    palette=dimension_palette,
    markers=True,
    errorbar=("ci", 95),
    marker="o",
    markersize=4,
    lw=1.25,
    ax=axes[2],
)
axes[2].axhline(1.0, color="black", lw=0.8)
axes[2].set_xscale("log", base=2)
axes[2].set_title(r"Ratio to the integral value")
axes[2].set_xlabel(r"System size $N$")
axes[2].set_ylabel(r"$K_{c,N}^{(\epsilon)}/K_c^{(\epsilon)}$")
axes[2].legend(title="Manifold dimension", loc="best", ncol=1, **legend_style)

for ax in axes:
    ax.title.set_fontsize(TITLE_SIZE)
    ax.xaxis.label.set_size(LABEL_SIZE)
    ax.yaxis.label.set_size(LABEL_SIZE)
    ax.tick_params(axis="both", labelsize=TICK_SIZE)

fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.92], w_pad=1.4)

labels = ["(a)", "(b)", "(c)"]
for label, ax in zip(labels, axes):
    bbox = ax.get_position()
    fig.text(
        bbox.x0,
        bbox.y1 + 0.022,
        label,
        ha="left",
        va="bottom",
        fontsize=PANEL_LABEL_SIZE,
        fontweight="normal",
    )

save_figure(fig, FIGURE_DIR / "fig02_finite_sample_threshold")
fig.savefig(FIGURE_DIR / "fig02_finite_sample_threshold_600dpi.png", dpi=600, bbox_inches="tight")